In [2]:
import math
import os
import time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import tiktoken
import torch
import torch.nn as nn
from rich import print
from torch.utils.data import DataLoader, Dataset

if torch.cuda.is_initialized():
    print(
        "[red]CUDA already initialised in this process, PYTORCH_CUDA_ALLOC_CONF "
        "set above has NO EFFECT, it only applies at context creation. Restart "
        "the kernel if you're chasing a fragmentation/OOM issue.[/red]"
    )

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
CONFIG_124m = {
    "vocab_size": 50257,
    "context_len": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
    "train_ctx": 1024,
}

STAGE2_CFG = {
    "base_weights": "/kaggle/input/models/berserkarch/vader/pytorch/default/1/vader.pth",
    "data_dir": "/kaggle/input/datasets/berserkarch/newone1/data/",
    "data_dir_sw": "/kaggle/input/datasets/berserkarch/star-wars/data/star_wars/",
    "sw_mix_ratio": 0.15,
    "out_dir": "/kaggle/working/out_stage2",
    "resume_from_input": None,

    "batch_size": 2,
    "grad_accum_steps": 64,

    "max_lr": 1.5e-4,
    "min_lr": 1.5e-5,
    "warmup_steps": 100,
    "max_steps": 3_000,

    "weight_decay": 0.1,
    "grad_clip": 1.0,
    "eval_every": 100,
    "eval_iters": 50,
    "ckpt_every": 250,
    "sample_prompt": "Once upon a time",
    "sample_temp": 0.6,
    "sample_top_k": 30,
    "log_every": 20,
}

LONG_PROBE = (
    "The Millennium Falcon shot through the asteroid field, engines screaming as it dodged "
    "chunks of rock the size of small moons. Han Solo gripped the controls tightly, sweat beading "
    "on his forehead as he weaved between obstacles that would have destroyed a lesser pilot. "
    "Behind him, Chewbacca roared a warning as another asteroid tumbled into their path. The Empire's "
    "pursuit ships had given up the chase minutes ago, unwilling to risk their own vessels in the "
    "chaotic debris field, but Han knew better than to relax just yet. Leia sat in the co-pilot's "
    "seat, studying a star chart that flickered with static, trying to plot a course that would "
    "take them clear of Imperial patrol routes without burning through what little fuel they had "
    "left. The hyperdrive had been acting up since their last engagement, and Han wasn't entirely "
    "sure it would hold together for another jump. Still, options were limited, and staying put "
    "in an asteroid field indefinitely wasn't one of them. Somewhere out there, the Rebellion needed "
    "the information they carried, and every hour spent"
)


# --------------------------------------------------------------------------
# Model — SDPA-fused attention (no manual mask buffer), fused GELU
# --------------------------------------------------------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.d_out = d_out
        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout_p = dropout

    def forward(self, x):
        b, num_tokens, _ = x.shape
        keys = self.w_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.w_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.w_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        context_vec = torch.nn.functional.scaled_dot_product_attention(
            queries, keys, values, is_causal=True,
            dropout_p=self.dropout_p if self.training else 0.0,
        )
        context_vec = context_vec.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_len"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        self.out_head.weight = self.tok_emb.weight

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


# --------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------
class MemmapDataset(Dataset):
    def __init__(self, bin_path, ctx_len):
        self.data = np.memmap(bin_path, dtype=np.uint16, mode="r")
        self.ctx_len = ctx_len

    def __len__(self):
        return (len(self.data) - 1) // self.ctx_len

    def __getitem__(self, idx):
        start = idx * self.ctx_len
        chunk = self.data[start : start + self.ctx_len + 1].astype(np.int64)
        chunk = torch.from_numpy(chunk)
        return chunk[:-1], chunk[1:]


def create_dataloader(bin_path, ctx_len, batch_size, shuffle):
    dataset = MemmapDataset(bin_path, ctx_len)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        drop_last=True, num_workers=2, pin_memory=True,
    )


# --------------------------------------------------------------------------
# Generation / sampling
# --------------------------------------------------------------------------
def generate(model, idx, max_new_tokens, context_size, temp=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cnd = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cnd)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits
            )
        if temp > 0.0:
            logits = logits / temp
            probs = torch.softmax(logits, dim=-1)
            idx_nxt = torch.multinomial(probs, num_samples=1)
        else:
            idx_nxt = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_nxt == eos_id:
            break
        idx = torch.cat((idx, idx_nxt), dim=1)
    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)


def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())


def generate_and_print_sample(model, tokenizer, device, prompt, ctx_len, temp=0.0, top_k=None):
    model.eval()
    encoded = text_to_token_ids(prompt, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate(model, encoded, max_new_tokens=60, context_size=ctx_len, temp=temp, top_k=top_k)
    print("[bold cyan]sample:[/bold cyan]", token_ids_to_text(token_ids, tokenizer).replace("\n", " "))
    model.train()


# --------------------------------------------------------------------------
# LR schedule
# --------------------------------------------------------------------------
def get_lr(step, cfg):
    if step < cfg["warmup_steps"]:
        return cfg["max_lr"] * (step + 1) / cfg["warmup_steps"]
    if step > cfg["max_steps"]:
        return cfg["min_lr"]
    decay_ratio = (step - cfg["warmup_steps"]) / (cfg["max_steps"] - cfg["warmup_steps"])
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return cfg["min_lr"] + coeff * (cfg["max_lr"] - cfg["min_lr"])


# --------------------------------------------------------------------------
# Eval
# --------------------------------------------------------------------------
@torch.no_grad()
def estimate_loss(model, loader, device, eval_iters):
    model.eval()
    losses = torch.zeros(eval_iters)
    it = iter(loader)
    for i in range(eval_iters):
        try:
            xb, yb = next(it)
        except StopIteration:
            it = iter(loader)
            xb, yb = next(it)
        xb, yb = xb.to(device), yb.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(xb)
            loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), yb.flatten())
        losses[i] = loss.item()
    model.train()
    return losses.mean().item()


# --------------------------------------------------------------------------
# Checkpointing
# --------------------------------------------------------------------------
def save_checkpoint(path, model, optimizer, scaler, step):
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "step": step,
    }, path)


def load_checkpoint(path, model, optimizer, scaler, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    return ckpt["step"]


def load_base_weights(model, path, device):
    raw_sd = torch.load(path, map_location=device)
    filtered_sd = {k: v for k, v in raw_sd.items() if not k.endswith(".att.mask")}
    missing, unexpected = model.load_state_dict(filtered_sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"unexpected mismatch loading base weights: missing={missing}, unexpected={unexpected}")
    dropped = len(raw_sd) - len(filtered_sd)
    print(f"[green]loaded base weights from {path} (dropped {dropped} legacy mask keys)[/green]")


# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------
def main():
    cfg = STAGE2_CFG
    assert cfg["sw_mix_ratio"] is not None, "set sw_mix_ratio from tokenize_star_wars.py's printed token count before running"

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Device in use: {device} (single-GPU, no DDP: see chat notes on Kaggle quota cost)")
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / 1e9
        reserved = torch.cuda.memory_reserved(device) / 1e9
        print(f"GPU memory before model init: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        if allocated > 0.5:
            print(
                "[red]non-trivial memory already in use before the model exists, "
                "this kernel likely has a previous run's tensors still alive, "
                "restart the kernel and rerun[/red]"
            )

    os.makedirs(cfg["out_dir"], exist_ok=True)
    tokenizer = tiktoken.get_encoding("gpt2")
    ctx_len = CONFIG_124m["train_ctx"]

    train_loader = create_dataloader(
        os.path.join(cfg["data_dir"], "train.bin"), ctx_len, cfg["batch_size"], shuffle=True
    )
    val_loader = create_dataloader(
        os.path.join(cfg["data_dir"], "validation.bin"), ctx_len, cfg["batch_size"], shuffle=False
    )
    sw_loader = create_dataloader(
        os.path.join(cfg["data_dir_sw"], "train.bin"), ctx_len, cfg["batch_size"], shuffle=True
    )

    model = Model(CONFIG_124m).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["max_lr"], weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda")

    ckpt_path = os.path.join(cfg["out_dir"], "checkpoint.pt")

    if cfg.get("resume_from_input") and not os.path.exists(ckpt_path):
        if os.path.exists(cfg["resume_from_input"]):
            import shutil
            shutil.copy(cfg["resume_from_input"], ckpt_path)
            print(f"[yellow]copied stage-2 checkpoint forward from {cfg['resume_from_input']}[/yellow]")
        else:
            print(
                f"[red]resume_from_input set but not found at {cfg['resume_from_input']}, "
                f"check the attached dataset slug, starting stage 2 from base weights instead[/red]"
            )

    if os.path.exists(ckpt_path):
        start_step = load_checkpoint(ckpt_path, model, optimizer, scaler, device)
        print(f"[yellow]resumed stage-2 at step {start_step}[/yellow]")
    else:
        load_base_weights(model, cfg["base_weights"], device)
        start_step = 0
        print("[green]starting stage 2 at step 0[/green]")

    train_iter = iter(train_loader)
    sw_iter = iter(sw_loader)
    val_log_path = os.path.join(cfg["out_dir"], "val_loss_log.csv")
    t0 = time.time()

    for step in range(start_step, cfg["max_steps"]):
        lr = get_lr(step, cfg)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        optimizer.zero_grad(set_to_none=True)
        accum_loss = 0.0
        for _ in range(cfg["grad_accum_steps"]):
            use_sw = torch.rand(1).item() < cfg["sw_mix_ratio"]
            active_iter = sw_iter if use_sw else train_iter
            try:
                xb, yb = next(active_iter)
            except StopIteration:
                if use_sw:
                    sw_iter = iter(sw_loader)
                    xb, yb = next(sw_iter)
                else:
                    train_iter = iter(train_loader)
                    xb, yb = next(train_iter)
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(xb)
                loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), yb.flatten())
                loss = loss / cfg["grad_accum_steps"]

            scaler.scale(loss).backward()
            accum_loss += loss.item()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
        scaler.step(optimizer)
        scaler.update()

        if step % cfg["log_every"] == 0:
            dt = time.time() - t0
            toks_per_step = cfg["batch_size"] * cfg["grad_accum_steps"] * ctx_len
            print(
                f"step {step:05d} | loss {accum_loss:.4f} | lr {lr:.2e} "
                f"| {dt / cfg['log_every']:.2f}s/step | ~{toks_per_step / max(dt / cfg['log_every'], 1e-9):.0f} tok/s"
            )
            t0 = time.time()

        if step % cfg["eval_every"] == 0 and step > 0:
            val_loss = estimate_loss(model, val_loader, device, cfg["eval_iters"])
            print(f"[bold green]step {step:05d} val_loss {val_loss:.4f}[/bold green]")
            with open(val_log_path, "a") as f:
                f.write(f"{step},{val_loss}\n")
            generate_and_print_sample(
                model, tokenizer, device, cfg["sample_prompt"], ctx_len,
                temp=cfg["sample_temp"], top_k=cfg["sample_top_k"],
            )
            generate_and_print_sample(
                model, tokenizer, device, LONG_PROBE, ctx_len,
                temp=cfg["sample_temp"], top_k=cfg["sample_top_k"],
            )

        if step % cfg["ckpt_every"] == 0 and step > 0:
            save_checkpoint(ckpt_path, model, optimizer, scaler, step + 1)
            print(f"[dim]checkpoint saved at step {step}[/dim]")

    save_checkpoint(ckpt_path, model, optimizer, scaler, cfg["max_steps"])
    torch.save(model.state_dict(), os.path.join(cfg["out_dir"], "vader_stage2_final.pth"))
    print("[bold green]stage 2 complete, final weights saved[/bold green]")


if __name__ == "__main__":
    main()

step 02800 | loss 3.2686 | lr 1.66e-05 | 11.92s/step | ~10994 tok/s

step 02800 val_loss 3.2348

sample: Once upon a time in the future, people could take a few days to recover from the disaster, but they were 
often unable to. The first step to recovery was to find a way to get the people to help. There were many people who
had recovered, but they had to find a way to get them to

sample: The Millennium Falcon shot through the asteroid field, engines screaming as it dodged chunks of rock the 
size of small moons. Han Solo gripped the controls tightly, sweat beading on his forehead as he weaved between 
obstacles that would have destroyed a lesser pilot. Behind him, Chewbacca roared a warning as another asteroid 
tumbled into their path. The Empire's pursuit ships had given up the chase minutes ago, unwilling to risk their own
vessels in the chaotic debris field, but Han knew better than to relax just yet. Leia sat in the co-pilot's seat, 
studying a star chart that flickered with static, trying to plot a course that would take them clear of Imperial 
patrol routes without burning through what little fuel they had left. The hyperdrive had been acting up since their
last engagement, and Han wasn't entirely sure it would hold together for another jump. Still, options were limited,
and staying put in an asteroid field indefinitely wasn't one of them. Somewhere out there, the Rebellion needed the
information they carried, and every hour spent on the ground was a good time.  The Rebellion's first foray into 
space was a desperate one, and the Rebel Alliance was already trying to use the Force to push the Rebels out of the
asteroid field. Han was on his way to take Han Solo and Leia Organa Solo to the moon

step 02820 | loss 3.3023 | lr 1.63e-05 | 12.18s/step | ~10759 tok/s

...

step 02920 | loss 3.2426 | lr 1.53e-05 | 12.20s/step | ~10742 tok/s

step 02940 | loss 3.1986 | lr 1.51e-05 | 11.94s/step | ~10980 tok/s

step 02960 | loss 3.1673 | lr 1.51e-05 | 11.94s/step | ~10976 tok/s

step 02980 | loss 3.1506 | lr 1.50e-05 | 11.95s/step | ~10970 tok/s

stage 2 complete, final weights saved